# Training of the credit risk assessment model (Credit Scoring Model)
**Version:** 3.1.0 (Updated)
This notebook downloads LendingClub data, cleans it, generates new features (Feature Engineering), and trains a ``RandomForest'' model with a search for optimal hyperparameters (`GridSearchCV`).

In [20]:
# --- MODULE 1: IMPORT LIBRARIES ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import joblib
import warnings

warnings.filterwarnings('ignore')
print("Libraries successfully loaded.")

Бібліотеки успішно завантажені.


In [21]:
# --- MODULE 2: LOADING DATA ---
# Specify the correct path to your extracted CSV file
# We add the letter 'r' before the line so that Windows correctly reads backslashes
file_path = r'../data/accepted_2007_to_2018Q4.csv' 
df = pd.read_csv(file_path, nrows=200000)

# Downloading only part of the data for speed (or remove nrows to download everything)
print("Loading data...")
df = pd.read_csv(file_path, nrows=200000) 
print(f"Loaded rows: {df.shape[0]}, columns: {df.shape[1]}")

Завантаження даних...
Завантажено рядків: 200000, колонок: 151


In [22]:
# --- MODULE 3: CLEANING AND FILTERING (DATA CLEANING) ---

# 1. We leave only completed credits (avoid data leakage from current ones)
valid_statuses = ['Fully Paid', 'Charged Off']
df = df[df['loan_status'].isin(valid_statuses)].copy()

# 2. Create a target variable (1 - Success, 0 - Default)
df['target'] = df['loan_status'].apply(lambda x: 1 if x == 'Fully Paid' else 0)

# 3. Filter anomalies (Outliers)
df = df[(df['annual_inc'] >= 1000) & (df['annual_inc'] <= 300000)]
df = df[df['dti'] <= 100]

# 4. We delete missing values ​​in critical columns
cols_to_check = ['emp_length', 'home_ownership', 'purpose', 'fico_range_low']
df = df.dropna(subset=cols_to_check)

print(f"Remaining records after cleaning: {df.shape[0]}")
print("Class balance:\n", df['target'].value_counts(normalize=True))

Залишилось записів після очищення: 163874
Баланс класів:
 target
1    0.803556
0    0.196444
Name: proportion, dtype: float64


In [23]:
# --- MODULE 4: FEATURE ENGINEERING ---

# 1. Convert 'term' (e.g. "36 months") into the number 36
df['term'] = df['term'].astype(str).str.extract('(\d+)').astype(float)

# 2. We create a powerful business metric: the ratio of loan amount to income
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)

print("Character engineering is complete.")

Інженерія ознак завершена.


In [24]:
# --- MODULE 5: DATA PREPARATION FOR ML (ENCODING & SPLITTING) ---

# We determine the final list of columns that are available to the bank BEFORE the loan is issued
features = [
    'annual_inc', 'loan_amnt', 'term', 'fico_range_low', 'dti', 'loan_to_income', 
    'emp_length', 'home_ownership', 'purpose'
]

X = df[features].copy()
y = df['target']

# We convert text columns into One-Hot Encoding format (0 and 1)
# drop_first=True reduces dimensionality and avoids multicollinearity
X = pd.get_dummies(X, columns=['emp_length', 'home_ownership', 'purpose'], drop_first=True)

# We save the list of final columns for the app.py application
final_columns = X.columns.tolist()
joblib.dump(final_columns, '../models/model_columns.pkl')
print(f"Number of features after One-Hot Encoding: {len(final_columns)}")

# Divide into training and test samples (Stratify preserves class proportions)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Breakdown complete: X_train shape:", X_train.shape)

Кількість фічей після One-Hot Encoding: 31
Розбиття завершено: X_train shape: (131099, 31)


In [25]:
# --- MODULE 6: MODEL TRAINING AND TUNING (GRID SEARCH) ---

# Basic model (we balance classes using class_weight)
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42)

# Hyperparameters for checking (you can expand the list, but it will increase the training time)
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [10, 15],
    'min_samples_split': [5, 10]
}

# Setting up cross-validation (3 folds)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# We initialize the search
grid_search = GridSearchCV(
    estimator=rf_base, 
    param_grid=param_grid, 
    cv=cv, 
    scoring='roc_auc', 
    n_jobs=-1, # Use all processor cores
    verbose=2
)

print("We start training. Wait...")
grid_search.fit(X_train, y_train)

# We keep the best model
best_rf_model = grid_search.best_estimator_
print(f"\nBest parameters found: {grid_search.best_params_}")

Починаємо навчання. Зачекайте...
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Найкращі знайдені параметри: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}


In [26]:
# --- MODULE 7: EVALUATION AND PRESERVATION ---

# We make predictions
y_pred = best_rf_model.predict(X_test)
y_pred_proba = best_rf_model.predict_proba(X_test)[:, 1]

# We derive the metrics
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# We save the model itself for the Flask application
joblib.dump(best_rf_model, '../models/credit_model.pkl')
print("Model successfully saved to file '../models/credit_model.pkl'!")

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.34      0.57      0.42      6438
           1       0.87      0.73      0.79     26337

    accuracy                           0.70     32775
   macro avg       0.61      0.65      0.61     32775
weighted avg       0.77      0.70      0.72     32775

ROC-AUC Score: 0.7058
Модель успішно збережена у файл 'credit_model.pkl'!
